# Agent Demo Notebook

Demonstrates each of the three agent nodes in the orchestrator pipeline:

| Section | Agent | Decision |
|---------|-------|---------|
| **1 — Observation agent** | `observation` | `accept` / `reject` |
| **2 — Weather agent** | `weather` | `use` / `wind_shift` / `stale` |
| **3 — Assimilation agent** | `assimilation` | `assimilate` / `inflate_and_assimilate` / `skip` / `reinitialize_from_obs` |

Each section shows:
- the **happy path** (normal case)
- one or more **edge cases** that flip the routing branch
- the **TwinState keys before and after** the node runs
- the **routing outcome** (which node would be next)

No FARSITE runs. All demos use `placeholder_nodes` or hand-crafted state dicts.


## 0. Setup

In [1]:
import sys, json, warnings
from datetime import datetime, timezone, timedelta
from pathlib import Path
import pprint

# ── repo root on sys.path ────────────────────────────────────────────────────
_repo = Path.cwd()
while _repo != _repo.parent:
    if (_repo / "orchestrator" / "__init__.py").exists():
        break
    _repo = _repo.parent
if str(_repo) not in sys.path:
    sys.path.insert(0, str(_repo))
if str(_repo / "src") not in sys.path:
    sys.path.insert(0, str(_repo / "src"))

from orchestrator.state import (
    ObservationEvent, OperatorRequest, ForecastPlan,
    AssimilationDecision, Signals, DecisionRecord,
    TwinState, initial_state, BASELINE, VSIZE_PRESETS,
)
from orchestrator import routing
from orchestrator.placeholders import placeholder_nodes, PlaceholderVerdict

UTC = timezone.utc
print("imports ok")


imports ok


In [2]:
# ── helper: pretty-print only the keys that changed ─────────────────────────
def show_diff(before: dict, after: dict):
    changed = {k: after[k] for k in after if after.get(k) != before.get(k)}
    if not changed:
        print("  (no keys changed)")
        return
    for k, v in changed.items():
        print(f"  {k}:")
        if hasattr(v, "model_dump"):
            pprint.pprint(v.model_dump(), indent=4)
        elif isinstance(v, list) and v and hasattr(v[-1], "model_dump"):
            pprint.pprint([x.model_dump() if hasattr(x, "model_dump") else x for x in v], indent=4)
        else:
            pprint.pprint(v, indent=4)

def show_routing(fn, state, label=""):
    result = fn(state)
    print(f"  routing.{fn.__name__}(state) → '{result}'" + (f"  ({label})" if label else ""))
    return result

# ── minimal shared fixtures ──────────────────────────────────────────────────
REQUEST = OperatorRequest(deadline_min=30.0)
T0  = datetime(2025, 1, 23, 12, 0, tzinfo=UTC)   # simulated observation time
T1  = datetime(2025, 1, 24,  6, 0, tzinfo=UTC)
T2  = datetime(2025, 1, 25,  3, 0, tzinfo=UTC)

def make_event(obs_time, feature_id="0", source="mapped_perimeter", injected=None):
    return ObservationEvent(
        obs_time=obs_time, arrival_time=obs_time + timedelta(minutes=5),
        source=source, path="data/border2_inputs/border_2_perimeters.geojson",
        feature_id=str(feature_id), injected=injected,
    )

def base_state(**overrides) -> TwinState:
    """Return a minimal TwinState for demos (no actual file I/O)."""
    s = TwinState(
        request=REQUEST,
        clock=T1,
        step=1,
        state_path="outputs/state.npz",
        prev_obs_path=None,
        pending=[make_event(T2)],
        current=make_event(T1, feature_id="1"),
        plan=None,
        ensemble_path=None,
        signals=None,
        obs_verdict=None,
        weather_verdict=None,
        assim_verdict=None,
        drift_history=[],
        cache={},
        log=[],
        metrics=[],
    )
    s.update(overrides)
    return s

print("fixtures ready")


fixtures ready


---
## 1 — Observation agent

**Job:** decide whether to trust the incoming perimeter observation.

**Verdict key:** `obs_verdict` — must have an `.action` of `"accept"` or `"reject"`.

**Routing outcome:** `routing.after_observation(state)` returns
- `"weather"` when `obs_verdict.action == "accept"`
- `"synthesize"` when `obs_verdict.action == "reject"` (a synthetic perimeter is grown instead)

The demos here use `placeholder_agent`, which always returns a fixed verdict. A real
observation agent would compute `Signals` (innovation, FIRMS agreement, containment check …)
and decide from those.


### 1a — Happy path: observation accepted

In [3]:
from orchestrator.placeholders import placeholder_agent

obs_agent_accept = placeholder_agent("observation", "obs_verdict", "accept")

state = base_state()
print("BEFORE:")
print(f"  obs_verdict = {state['obs_verdict']}")
print(f"  current.source = {state['current'].source}")

delta = obs_agent_accept(state)
after = {**state, **delta}

print("\nAFTER (changed keys):")
show_diff(state, after)

print("\nROUTING:")
show_routing(routing.after_observation, after, "normal case → continue to weather")


BEFORE:
  obs_verdict = None
  current.source = mapped_perimeter

AFTER (changed keys):
  obs_verdict:
{'action': 'accept', 'reason': 'placeholder'}
  log:
[   {   'agent_latency_s': 0.0,
        'decision': {'action': 'accept', 'reason': 'placeholder'},
        'inputs': {   'event': {   'arrival_time': '2025-01-24T06:05:00Z',
                                   'feature_id': '1',
                                   'injected': None,
                                   'obs_time': '2025-01-24T06:00:00Z',
                                   'path': 'data/border2_inputs/border_2_perimeters.geojson',
                                   'source': 'mapped_perimeter'}},
        'node': 'observation',
        'overridden_by_guardrail': False,
        'policy': 'placeholder',
        'sim_time': datetime.datetime(2025, 1, 24, 6, 0, tzinfo=datetime.timezone.utc),
        'tokens': 0}]

ROUTING:
  routing.after_observation(state) → 'weather'  (normal case → continue to weather)


'weather'

### 1b — Edge case: observation rejected → routing goes to `synthesize`

In [4]:
obs_agent_reject = placeholder_agent("observation", "obs_verdict", "reject")

state = base_state()
delta = obs_agent_reject(state)
after = {**state, **delta}

print("AFTER (changed keys):")
show_diff(state, after)

print("\nROUTING:")
show_routing(routing.after_observation, after, "rejected → synthesize branch")


AFTER (changed keys):
  obs_verdict:
{'action': 'reject', 'reason': 'placeholder'}
  log:
[   {   'agent_latency_s': 0.0,
        'decision': {'action': 'reject', 'reason': 'placeholder'},
        'inputs': {   'event': {   'arrival_time': '2025-01-24T06:05:00Z',
                                   'feature_id': '1',
                                   'injected': None,
                                   'obs_time': '2025-01-24T06:00:00Z',
                                   'path': 'data/border2_inputs/border_2_perimeters.geojson',
                                   'source': 'mapped_perimeter'}},
        'node': 'observation',
        'overridden_by_guardrail': False,
        'policy': 'placeholder',
        'sim_time': datetime.datetime(2025, 1, 24, 6, 0, tzinfo=datetime.timezone.utc),
        'tokens': 0}]

ROUTING:
  routing.after_observation(state) → 'synthesize'  (rejected → synthesize branch)


'synthesize'

### 1c — Edge case: smart placeholder rejects injected observations

A real agent would use `Signals` to decide. Here we show a simple rule-based placeholder
that rejects any observation flagged as `injected` (a fault injection used in tests).


In [5]:
def reject_if_injected(state: TwinState) -> str:
    """Reject observations that have been deliberately corrupted for testing."""
    current = state.get("current")
    if current and current.injected is not None:
        return "reject"
    return "accept"

obs_agent_smart = placeholder_agent("observation", "obs_verdict", reject_if_injected)

# Case A: clean observation
state_clean = base_state(current=make_event(T1, injected=None))
delta = obs_agent_smart(state_clean)
print("Clean obs →", dict(delta)["obs_verdict"].action)
show_routing(routing.after_observation, {**state_clean, **delta})

# Case B: injected/corrupt observation
state_bad = base_state(current=make_event(T1, injected="shifted_2km"))
delta = obs_agent_smart(state_bad)
print("\nInjected obs →", dict(delta)["obs_verdict"].action)
show_routing(routing.after_observation, {**state_bad, **delta})


Clean obs → accept
  routing.after_observation(state) → 'weather'

Injected obs → reject
  routing.after_observation(state) → 'synthesize'


'synthesize'

---
## 2 — Weather agent

**Job:** fetch (or confirm) the weather for the forecast interval and decide how to handle it.

**Verdict key:** `weather_verdict` — `.action` is typically `"use"`, but can flag
`"wind_shift"` (triggers ensemble inflation) or `"stale"` (use fallback values).

**Routing outcome:** `routing.after_observation` already decided we're here (obs was accepted).
The weather agent does not branch the graph itself — routing continues to `forecast` after it.
The verdict is consumed by the **assimilation agent** (Section 3) to decide whether to inflate.

The agent also caches the fetched weather in `cache["weather_last"]` so the forecast step
can pass it to FARSITE without re-fetching.


### 2a — Happy path: stable weather, verdict `'use'`

In [6]:
from orchestrator.placeholders import placeholder_weather

FALLBACK_WEATHER = {"wind_speeds": [10.0], "wind_directions": [225.0], "used_fallback": True}
REAL_WEATHER     = {"wind_speeds": [8.0, 11.0, 9.5], "wind_directions": [220.0, 230.0, 225.0],
                    "used_fallback": False}

weather_agent = placeholder_weather(weather=REAL_WEATHER)

state = base_state(obs_verdict=PlaceholderVerdict(action="accept"))
delta = weather_agent(state)
after = {**state, **delta}

print("AFTER (changed keys):")
show_diff(state, after)
print("\ncache['weather_last']:", after["cache"].get("weather_last"))


AFTER (changed keys):
  weather_verdict:
{'action': 'use', 'reason': 'placeholder'}
  cache:
{   'weather_last': {   'used_fallback': False,
                        'wind_directions': [220.0, 230.0, 225.0],
                        'wind_speeds': [8.0, 11.0, 9.5]}}
  log:
[   {   'agent_latency_s': 0.0,
        'decision': {'action': 'use', 'reason': 'placeholder'},
        'inputs': {   'event': {   'arrival_time': '2025-01-24T06:05:00Z',
                                   'feature_id': '1',
                                   'injected': None,
                                   'obs_time': '2025-01-24T06:00:00Z',
                                   'path': 'data/border2_inputs/border_2_perimeters.geojson',
                                   'source': 'mapped_perimeter'}},
        'node': 'weather',
        'overridden_by_guardrail': False,
        'policy': 'placeholder',
        'sim_time': datetime.datetime(2025, 1, 24, 6, 0, tzinfo=datetime.timezone.utc),
        'tokens': 0}]

cache

### 2b — Edge case: single fallback reading (only one wind observation)

In [7]:
weather_agent_fallback = placeholder_weather(weather=FALLBACK_WEATHER)

state = base_state(obs_verdict=PlaceholderVerdict(action="accept"))
delta = weather_agent_fallback(state)
after = {**state, **delta}

print("weather cached:")
pprint.pprint(after["cache"].get("weather_last"))
print("\nused_fallback:", after["cache"]["weather_last"]["used_fallback"])
print("(a real agent would note this in the verdict and the assimilation agent")
print(" would increase wsize_m to trust the forecast more)")


weather cached:
{'used_fallback': True, 'wind_directions': [225.0], 'wind_speeds': [10.0]}

used_fallback: True
(a real agent would note this in the verdict and the assimilation agent
 would increase wsize_m to trust the forecast more)


### 2c — Edge case: wind-shift verdict

A real weather agent compares the forecast-interval wind with the previous interval.
Here we show what a wind-shift verdict looks like in state, and how the assimilation
agent (Section 3) uses it.


In [8]:
# Craft a wind-shift verdict by hand (what a real weather agent would produce)
wind_shift_verdict = PlaceholderVerdict(action="wind_shift",
                                        reason="direction rotated 95° since last step")

state = base_state(obs_verdict=PlaceholderVerdict(action="accept"),
                   weather_verdict=wind_shift_verdict,
                   cache={"weather_last": REAL_WEATHER})

print("weather_verdict.action =", state["weather_verdict"].action)
print("\nEffect: the assimilation agent (Section 3) should respond with")
print("  action='inflate_and_assimilate' — see demo 3c.")


weather_verdict.action = wind_shift

Effect: the assimilation agent (Section 3) should respond with
  action='inflate_and_assimilate' — see demo 3c.


---
## 3 — Assimilation agent

**Job:** decide *how* to update the fire state given the forecast ensemble and the observation.

**Verdict key:** `assim_verdict` — an `AssimilationDecision` with four possible actions:

| Action | When to use |
|--------|-------------|
| `assimilate` | Normal EnKF update. Observation is consistent with the forecast. |
| `inflate_and_assimilate` | Ensemble is too tight (after a wind shift, or after several steps of agreement). Widens the spread first. |
| `skip` | Observation is wrong (all independent checks failed). Do not update the state. |
| `reinitialize_from_obs` | Model has drifted badly for several cycles. Restart the state from the observation. |

The agent also chooses `vsize_m` (how much to trust the observation) and `wsize_m`
(how much to trust the forecast). `VSIZE_PRESETS` gives sensible starting values by source.

**Routing outcome:** `routing.after_update` runs *after* `update` (the plain step that
saves things), not after the assimilation agent directly. It returns `"receive"` while
observations remain, `"end"` when the replay is finished.


### 3a — Happy path: normal `assimilate`

In [9]:
from orchestrator.state import AssimilationDecision

assim_agent_normal = placeholder_agent("assimilation", "assim_verdict", "assimilate")

state = base_state(
    obs_verdict     = PlaceholderVerdict(action="accept"),
    weather_verdict = PlaceholderVerdict(action="use"),
    ensemble_path   = "placeholder_ensemble_step1.npz",
)
delta = assim_agent_normal(state)
after = {**state, **delta}

print("AFTER (changed keys):")
show_diff(state, after)

print("\nIn a real run the assimilation step would call:")
print("  adapter.enkf_update(ensemble, obs_vec,")
print(f"    vsize_m={VSIZE_PRESETS['mapped_perimeter']},  # from VSIZE_PRESETS")
print(f"    wsize_m={BASELINE['wsize_m']},               # from BASELINE")
print("    inflation=1.0)")


AFTER (changed keys):
  assim_verdict:
{'action': 'assimilate', 'reason': 'placeholder'}
  log:
[   {   'agent_latency_s': 0.0,
        'decision': {'action': 'assimilate', 'reason': 'placeholder'},
        'inputs': {   'event': {   'arrival_time': '2025-01-24T06:05:00Z',
                                   'feature_id': '1',
                                   'injected': None,
                                   'obs_time': '2025-01-24T06:00:00Z',
                                   'path': 'data/border2_inputs/border_2_perimeters.geojson',
                                   'source': 'mapped_perimeter'}},
        'node': 'assimilation',
        'overridden_by_guardrail': False,
        'policy': 'placeholder',
        'sim_time': datetime.datetime(2025, 1, 24, 6, 0, tzinfo=datetime.timezone.utc),
        'tokens': 0}]

In a real run the assimilation step would call:
  adapter.enkf_update(ensemble, obs_vec,
    vsize_m=200,  # from VSIZE_PRESETS
    wsize_m=100.0,               # from B

### 3b — Edge case: skip (observation fails all independent checks)

In [10]:
# Craft a Signals object where all independent checks fail
bad_signals = Signals(
    normalized_innovation=1.2,     # not unusually high — forecast-based check passes
    agrees_with_firms=False,       # no satellite hotspots inside the perimeter
    contains_previous=False,       # perimeter shrank (fire doesn't shrink)
    growth_plausible=False,        # implied spread rate is physically impossible
    obs_age_min=25.0,
    source="mapped_perimeter",
    wind_shift=False,
    consecutive_disagreements=0,
)

print("independent_check_failed:", bad_signals.independent_check_failed)
print("  (computed from: agrees_with_firms=False OR contains_previous=False OR growth_plausible=False)")
print()

# A rules-based agent would see this and choose 'skip'
skip_verdict = AssimilationDecision(
    action="skip",
    vsize_m=VSIZE_PRESETS["mapped_perimeter"],
    wsize_m=BASELINE["wsize_m"],
    reason="all three forecast-independent checks failed; "
           "observation is likely wrong",
)

state = base_state(
    obs_verdict=PlaceholderVerdict(action="accept"),
    signals=bad_signals,
    ensemble_path="placeholder_ensemble_step1.npz",
)
state["assim_verdict"] = skip_verdict

print("assim_verdict:")
pprint.pprint(skip_verdict.model_dump())
print("\nEffect: the update step will NOT save a new state_path.")
print("The fire state carries forward unchanged from the previous step.")


independent_check_failed: True
  (computed from: agrees_with_firms=False OR contains_previous=False OR growth_plausible=False)

assim_verdict:
{'action': 'skip',
 'inflation': 1.0,
 'reason': 'all three forecast-independent checks failed; observation is '
           'likely wrong',
 'vsize_m': 200.0,
 'wsize_m': 100.0}

Effect: the update step will NOT save a new state_path.
The fire state carries forward unchanged from the previous step.


### 3c — Edge case: `inflate_and_assimilate` after a wind shift

In [11]:
inflate_verdict = AssimilationDecision(
    action="inflate_and_assimilate",
    vsize_m=VSIZE_PRESETS["mapped_perimeter"],
    wsize_m=BASELINE["wsize_m"] * 1.5,     # trust the forecast less after a wind shift
    inflation=1.4,                           # widen the ensemble spread by 40 %
    reason="wind direction rotated 95° since last step; "
           "ensemble under-represents the new spread pattern",
)

state = base_state(
    obs_verdict     = PlaceholderVerdict(action="accept"),
    weather_verdict = PlaceholderVerdict(action="wind_shift"),
    ensemble_path   = "placeholder_ensemble_step1.npz",
)
state["assim_verdict"] = inflate_verdict

print("assim_verdict:")
pprint.pprint(inflate_verdict.model_dump())
print()
print("inflation=", inflate_verdict.inflation,
      " → cov_X will be multiplied by", inflate_verdict.inflation**2,
      "before the EnKF update")


assim_verdict:
{'action': 'inflate_and_assimilate',
 'inflation': 1.4,
 'reason': 'wind direction rotated 95° since last step; ensemble '
           'under-represents the new spread pattern',
 'vsize_m': 200.0,
 'wsize_m': 150.0}

inflation= 1.4  → cov_X will be multiplied by 1.9599999999999997 before the EnKF update


### 3d — Edge case: `reinitialize_from_obs` after persistent drift

In [12]:
# Simulate a drift history where the forecast has missed high for 4 steps in a row
long_drift = [2.1, 2.5, 2.8, 3.2]   # normalized innovations (> 1 = surprising)

reinit_verdict = AssimilationDecision(
    action="reinitialize_from_obs",
    vsize_m=VSIZE_PRESETS["mapped_perimeter"],
    wsize_m=BASELINE["wsize_m"],
    reason=f"4 consecutive high innovations (last: {long_drift[-1]:.1f}); "
            "model has systematically underforecast; restarting from observation",
)

state = base_state(
    obs_verdict     = PlaceholderVerdict(action="accept"),
    weather_verdict = PlaceholderVerdict(action="use"),
    ensemble_path   = "placeholder_ensemble_step4.npz",
    drift_history   = long_drift,
)
state["assim_verdict"] = reinit_verdict

print("drift_history:", state["drift_history"])
print()
print("assim_verdict:")
pprint.pprint(reinit_verdict.model_dump())
print()
print("Effect: state_vec is set directly from the observation perimeter.")
print("cov_X is reset to the initial prior covariance (re-estimates uncertainty).")


drift_history: [2.1, 2.5, 2.8, 3.2]

assim_verdict:
{'action': 'reinitialize_from_obs',
 'inflation': 1.0,
 'reason': '4 consecutive high innovations (last: 3.2); model has '
           'systematically underforecast; restarting from observation',
 'vsize_m': 200.0,
 'wsize_m': 100.0}

Effect: state_vec is set directly from the observation perimeter.
cov_X is reset to the initial prior covariance (re-estimates uncertainty).


---
## 4 — Full mini-replay (all three agents, two cycles)

Runs the complete pipeline with two synthetic observation events, capturing the
step trace, metrics, and decision log — without any FARSITE.

Cycle 1: observation accepted → normal assimilation
Cycle 2: observation rejected → synthetic perimeter grown → assimilation skips
         (since there's no real ensemble to update from)


In [13]:
import functools
from orchestrator.graph import build_pipeline, run_pipeline, memory_checkpointer
from orchestrator.steps import receive, update
from orchestrator.state import DecisionRecord, ObservationEvent

# Two events: second one will be deliberately rejected
events = [
    make_event(T1, feature_id="1"),
    make_event(T2, feature_id="2", injected="shifted_2km"),
]

def reject_if_injected_agent(state: TwinState) -> dict:
    action = "reject" if (state.get("current") and state["current"].injected) else "accept"
    verdict = PlaceholderVerdict(action=action)
    record  = DecisionRecord(
        sim_time=state["clock"], node="observation", policy="demo",
        inputs={"injected": state["current"].injected if state.get("current") else None},
        decision=verdict.model_dump(), overridden_by_guardrail=False, agent_latency_s=0.0,
    )
    return {"obs_verdict": verdict, "log": [record]}

def weather_agent(state: TwinState) -> dict:
    verdict = PlaceholderVerdict(action="use")
    record  = DecisionRecord(
        sim_time=state["clock"], node="weather", policy="demo",
        inputs={}, decision=verdict.model_dump(), overridden_by_guardrail=False, agent_latency_s=0.0,
    )
    return {"weather_verdict": verdict,
            "cache": {**state["cache"], "weather_last": REAL_WEATHER},
            "log": [record]}

def assim_agent(state: TwinState) -> dict:
    # skip if we came through synthesize (no real ensemble)
    is_synthetic = state.get("current") and state["current"].source == "synthetic_perimeter"
    action = "skip" if is_synthetic else "assimilate"
    verdict = PlaceholderVerdict(action=action)
    record  = DecisionRecord(
        sim_time=state["clock"], node="assimilation", policy="demo",
        inputs={"is_synthetic": is_synthetic},
        decision=verdict.model_dump(), overridden_by_guardrail=False, agent_latency_s=0.0,
    )
    return {"assim_verdict": verdict, "log": [record]}

def forecast_placeholder(state: TwinState) -> dict:
    from orchestrator.state import ForecastPlan, BASELINE
    plan = ForecastPlan(ensemble_size=BASELINE["ensemble_size"],
                        resolution_m=BASELINE["resolution_m"],
                        workers=BASELINE["workers"],
                        reason="demo placeholder")
    return {"plan": plan, "ensemble_path": f"demo_ensemble_step{state['step']}.npz"}

def synthesize_stub(state: TwinState) -> dict:
    """Demo stub: marks current as synthetic without reading any file from disk.
    The real synthesize (steps.py) calls make_synthetic_event which reads the GeoJSON;
    that requires the actual data files. For the demo we just tag the event."""
    rejected = state["current"]
    # Build a synthetic event that mirrors the rejected one but marks it as synthetic
    synthetic = ObservationEvent(
        obs_time=rejected.obs_time,
        arrival_time=rejected.arrival_time,
        source="synthetic_perimeter",
        path=rejected.path,          # still points to the same file (not read in demo)
        feature_id=rejected.feature_id,
        injected=rejected.injected,
    )
    record = DecisionRecord(
        sim_time=state["clock"], node="synthesize", policy="stub",
        inputs={"rejected_source": rejected.source, "injected": rejected.injected},
        decision={"synthetic": synthetic.model_dump(mode="json"),
                  "reason": "demo stub — no file I/O"},
        overridden_by_guardrail=False, agent_latency_s=0.0,
    )
    return {"current": synthetic, "log": [record]}

nodes = {
    "receive":      receive,
    "observation":  reject_if_injected_agent,
    "synthesize":   synthesize_stub,          # stub: no disk I/O
    "weather":      weather_agent,
    "forecast":     forecast_placeholder,
    "assimilation": assim_agent,
    "update":       update,
}

graph = build_pipeline(nodes)  # no checkpointer needed for the demo
print("Graph compiled.")


Graph compiled.


In [14]:
import warnings
warnings.filterwarnings("ignore")  # suppress LangGraph checkpoint serialization warnings

start_state = initial_state(events, REQUEST, state_path="outputs/demo_state.npz")
final = run_pipeline(graph, start_state)
print("Replay finished. Steps:", final["step"])


Replay finished. Steps: 2


In [15]:
import pandas as pd

print("=== Metrics (one row per cycle) ===")
display(pd.DataFrame(final["metrics"]))

print("\n=== Decision log ===")
log_rows = [
    {
        "step":   r.step if hasattr(r, "step") else None,
        "sim_time": r.sim_time.strftime("%m-%d %H:%M"),
        "node":   r.node,
        "policy": r.policy,
        "action": r.decision.get("action") if isinstance(r.decision, dict) else None,
        "reason": r.decision.get("reason") if isinstance(r.decision, dict) else None,
    }
    for r in final["log"]
]
display(pd.DataFrame(log_rows))


=== Metrics (one row per cycle) ===


,step,clock,obs_time,source,synthetic,obs_action,weather_action,assim_action
0,1,2025-01-24T06:05:00+00:00,2025-01-24T06:00:00+00:00,mapped_perimeter,False,accept,use,assimilate
1,2,2025-01-25T03:05:00+00:00,2025-01-25T03:00:00+00:00,synthetic_perimeter,True,reject,use,skip



=== Decision log ===


,step,sim_time,node,policy,action,reason
0,None,01-24 06:05,observation,demo,accept,placeholder
1,None,01-24 06:05,weather,demo,use,placeholder
2,None,01-24 06:05,assimilation,demo,assimilate,placeholder
3,None,01-25 03:05,observation,demo,reject,placeholder
4,None,01-25 03:05,synthesize,stub,NaN,demo stub — no file I/O
5,None,01-25 03:05,weather,demo,use,placeholder
6,None,01-25 03:05,assimilation,demo,skip,placeholder


In [16]:
print("=== Routing summary ===")
for m in final["metrics"]:
    obs_a   = m["obs_action"]    or "—"
    wea_a   = m["weather_action"] or "—"
    assim_a = m["assim_action"]  or "—"
    syn     = " [SYNTHETIC]" if m["synthetic"] else ""
    print(f"  cycle {m['step']}  source={m['source']}{syn}  "
          f"obs={obs_a}  weather={wea_a}  assim={assim_a}")


=== Routing summary ===
  cycle 1  source=mapped_perimeter  obs=accept  weather=use  assim=assimilate
  cycle 2  source=synthetic_perimeter [SYNTHETIC]  obs=reject  weather=use  assim=skip
